In [7]:
import os
import torch
from utils.rope import compute_rope_cos_sin, inverse_rope

# load all environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  For a RULER subtask:
```
  python lm_eval_script.py \
    -m meta-llama/Llama-3.1-8B-Instruct \
    -t ruler_vt \
    --limit 1 \
    -kc baseline \
    -vc baseline \
    --dump_full_kv_dir ./results/kv_dumps
```
  For a LongBench subtask:
```
  python lm_eval_script.py \
    -m meta-llama/Llama-3.1-8B-Instruct \
    -t longbench_qasper \
    --limit 1 \
    -kc baseline \
    -vc baseline \
    --dump_full_kv_dir ./results/kv_dumps
```

# Load KVs

In [8]:
def unrope_dumped_keys(keys: torch.Tensor, rope_theta: float) -> torch.Tensor:
    seq_len = keys.shape[-2]
    head_dim = keys.shape[-1]
    cos, sin = compute_rope_cos_sin(
        seq_len,
        head_dim,
        rope_theta,
        keys.device,
        keys.dtype,
    )
    return inverse_rope(keys, cos, sin)

kv_dump_dir = "./results/kv_dumps/meta-llama_Llama-3.1-8B-Instruct/niah_multiquery_raw_kv.pt"
kvs = torch.load(kv_dump_dir)
print(kvs.keys())

raw_keys = kvs["keys"]
rope_theta = float(kvs.get("rope_theta", 500_000.0))
keys = unrope_dumped_keys(raw_keys, rope_theta)
values = kvs["values"]
prompt_len = int(kvs.get("prompt_len", keys.shape[-2]))

keys.shape, values.shape, prompt_len, rope_theta

dict_keys(['task_name', 'input_ids', 'prompt_len', 'model_name', 'keys', 'values'])


(torch.Size([32, 1, 8, 3770, 128]),
 torch.Size([32, 1, 8, 3770, 128]),
 3770,
 500000.0)

In [9]:
from IPython.display import display

from utils.matrix_decomposition import (
    DECOMP_METHODS,
    decompose_grouped_xkv_to_segment_store,
    decompose_to_segment_store,
    reconstruct_segments,
)
from utils.segmentation import (
    build_cluster_segment_ranges,
    group_keys_by_cluster,
    group_sequences_by_cluster,
    kmeans_cluster_sequences,
)


def _ensure_layer_batched_keys(keys):
    if keys.dim() == 4:
        return keys.unsqueeze(1)
    if keys.dim() == 5:
        return keys
    raise ValueError(
        "Expected keys with shape [layers, heads, seq, dim] or "
        "[layers, batch, heads, seq, dim]."
    )

def _resolve_kmeans_dtype(kmeans_dtype):
    if isinstance(kmeans_dtype, str):
        try:
            kmeans_dtype = getattr(torch, kmeans_dtype)
        except AttributeError as exc:
            raise ValueError(f"Unknown kmeans dtype: {kmeans_dtype}") from exc
    if not isinstance(kmeans_dtype, torch.dtype):
        raise TypeError("kmeans_dtype must be a torch.dtype or dtype name.")
    return kmeans_dtype

def _get_cluster_count(seq_len, n_clusters, kmeans_cluster_size=None):
    if seq_len == 0:
        return 0
    if kmeans_cluster_size is not None:
        cluster_count = int(round(seq_len / kmeans_cluster_size))
    else:
        cluster_count = n_clusters
    return max(1, min(cluster_count, seq_len))

def _select_prefix(keys, prefix_end=None, local_window=0):
    seq_len = keys.size(-2)
    if prefix_end is None:
        prefix_end = seq_len
    if prefix_end < 0 or prefix_end > seq_len:
        raise ValueError(
            "prefix_end must be between 0 and the sequence length."
        )
    suffix_start = max(0, prefix_end - local_window)
    return keys[..., :suffix_start, :], suffix_start

def _validate_cluster_axis(cluster_axis):
    if cluster_axis not in {"rows", "cols"}:
        raise ValueError("cluster_axis must be 'rows' or 'cols'.")

def _group_features_and_ranges(features, assignments, n_clusters):
    grouped_features, _, _ = group_sequences_by_cluster(features, assignments)
    segment_ranges = build_cluster_segment_ranges(
        assignments,
        n_clusters=n_clusters,
    )
    return grouped_features, segment_ranges

def _cluster_items_and_ranges(
    features,
    n_clusters,
    kmeans_n_iter,
    kmeans_init,
    kmeans_dtype,
    kmeans_cluster_size=None,
    cluster_axis="rows",
):
    _validate_cluster_axis(cluster_axis)
    item_matrix = (
        features
        if cluster_axis == "rows"
        else features.transpose(1, 2).contiguous()
    )
    cluster_count = _get_cluster_count(
        item_matrix.size(1),
        n_clusters,
        kmeans_cluster_size=kmeans_cluster_size,
    )
    assignments = kmeans_cluster_sequences(
        item_matrix,
        n_clusters=cluster_count,
        n_iter=max(1, kmeans_n_iter),
        kmeans_init=kmeans_init,
        dtype=kmeans_dtype,
    )
    grouped_items, segment_ranges = _group_features_and_ranges(
        item_matrix,
        assignments,
        cluster_count,
    )
    return item_matrix, grouped_items, assignments, segment_ranges, cluster_count

def _scatter_from_grouped(grouped_features, segment_ranges):
    metrics = []
    for batch_idx, batch_ranges in enumerate(segment_ranges):
        batch_features = grouped_features[batch_idx]
        frob_sq = batch_features.pow(2).sum()
        scatter = batch_features.new_zeros(())
        for start_idx, end_idx in batch_ranges:
            cluster = batch_features[start_idx:end_idx]
            centroid = cluster.mean(dim=0, keepdim=True)
            scatter = scatter + (cluster - centroid).pow(2).sum()

        denom = float(frob_sq.item())
        scatter_value = float(scatter.item())
        metrics.append(
            {
                "J_kmeans": scatter_value,
                "eta": scatter_value / denom if denom > 0 else 0.0,
            }
        )
    return metrics

def _group_tensor_last_dim_by_cluster(tensor, assignments):
    if assignments.shape != (tensor.size(0), tensor.size(-1)):
        raise ValueError(
            f"Expected assignments shape {(tensor.size(0), tensor.size(-1))}, "
            f"got {tuple(assignments.shape)}."
        )
    permutation = torch.argsort(assignments, dim=-1)
    inverse_permutation = torch.empty_like(permutation)
    original_positions = (
        torch.arange(
            permutation.size(-1),
            device=permutation.device,
            dtype=permutation.dtype,
        )
        .unsqueeze(0)
        .expand_as(permutation)
    )
    inverse_permutation.scatter_(1, permutation, original_positions)
    gather_shape = (tensor.size(0),) + (1,) * (tensor.dim() - 2) + (tensor.size(-1),)
    gather_idx = permutation.view(gather_shape).expand_as(tensor)
    grouped_tensor = torch.gather(tensor, dim=tensor.dim() - 1, index=gather_idx)
    return grouped_tensor, permutation, inverse_permutation

def _restore_tensor_last_dim(grouped_tensor, inverse_permutation):
    gather_shape = (grouped_tensor.size(0),) + (1,) * (grouped_tensor.dim() - 2) + (grouped_tensor.size(-1),)
    gather_idx = inverse_permutation.view(gather_shape).expand_as(grouped_tensor)
    return torch.gather(
        grouped_tensor,
        dim=grouped_tensor.dim() - 1,
        index=gather_idx,
    )

def _get_decomposition(
    decomposition_method="svd",
    rank_selection="comp_ratio",
    comp_ratio=2.0,
    energy_threshold=0.95,
    decomp_n_iter=3,
    decomp_lr=1e-2,
):
    if decomposition_method not in DECOMP_METHODS:
        raise ValueError(
            f"Unknown decomposition_method: {decomposition_method}. "
            f"Available methods: {sorted(DECOMP_METHODS)}"
        )
    return DECOMP_METHODS[decomposition_method], {
        "rank_selection": rank_selection,
        "cr": comp_ratio,
        "energy_threshold": energy_threshold,
        "n_iter": decomp_n_iter,
        "lr": decomp_lr,
        "quantise_a": False,
        "quantise_b": False,
        "compressor_bits": 4,
    }

def _low_rank_recon_metrics(original, reconstructed):
    if original.shape != reconstructed.shape:
        raise ValueError(
            f"Shape mismatch: {original.shape} vs {reconstructed.shape}"
        )
    orig_flat = original.reshape(original.size(0), -1)
    recon_flat = reconstructed.reshape(reconstructed.size(0), -1)
    numer = (orig_flat - recon_flat).pow(2).sum(dim=-1).sqrt()
    denom = orig_flat.pow(2).sum(dim=-1).sqrt()
    metrics = []
    for idx in range(original.size(0)):
        denom_value = float(denom[idx].item())
        error_value = float(numer[idx].item())
        metrics.append(
            {
                "low_rank_recon_error": error_value,
                "relative_low_rank_recon_error": (
                    error_value / denom_value if denom_value > 0 else 0.0
                ),
            }
        )
    return metrics

def _merge_metric_lists(*metric_lists):
    num_items = len(metric_lists[0])
    return [
        {
            key: value
            for metric_dict in metric_dicts
            for key, value in metric_dict.items()
        }
        for metric_dicts in zip(*metric_lists)
    ]

def _reconstruct_lr_segments(
    grouped_tensor,
    segment_ranges,
    decompose_fn,
    decomp_kwargs,
    cluster_axis="rows",
):
    _validate_cluster_axis(cluster_axis)
    tensor_to_decompose = (
        grouped_tensor
        if cluster_axis == "rows"
        else grouped_tensor.transpose(-2, -1).contiguous()
    )
    layer_segments = decompose_to_segment_store(
        tensor_to_decompose,
        decompose_fn,
        segment_ranges=segment_ranges,
        **decomp_kwargs,
    )
    reconstructed = reconstruct_segments(
        layer_segments,
        tensor_to_decompose[..., :0, :],
    )
    return (
        reconstructed
        if cluster_axis == "rows"
        else reconstructed.transpose(-2, -1).contiguous()
    )

def _reconstruct_xkv_segments(grouped_tensor, segment_ranges, decomp_kwargs, cluster_axis="rows"):
    _validate_cluster_axis(cluster_axis)
    tensor_to_decompose = (
        grouped_tensor
        if cluster_axis == "rows"
        else grouped_tensor.transpose(-2, -1).contiguous()
    )
    grouped_segments = decompose_grouped_xkv_to_segment_store(
        tensor_to_decompose.unsqueeze(1),
        segment_ranges=segment_ranges,
        **decomp_kwargs,
    )
    reconstructed = reconstruct_segments(
        grouped_segments,
        tensor_to_decompose.unsqueeze(1)[..., :0, :],
    ).squeeze(1)
    return (
        reconstructed
        if cluster_axis == "rows"
        else reconstructed.transpose(-2, -1).contiguous()
    )


def analyze_kmeans_lrk(
    keys,
    n_clusters=8,
    kmeans_cluster_size=None,
    kmeans_n_iter=8,
    kmeans_init="infllm",
    kmeans_dtype=torch.float32,
    kmeans_mode="per_head",
    prefix_end=None,
    local_window=0,
    include_head_breakdown=False,
    decomposition_method="svd",
    rank_selection="comp_ratio",
    comp_ratio=2.0,
    energy_threshold=0.95,
    decomp_n_iter=3,
    decomp_lr=1e-2,
    cluster_axis="rows",
):
    keys = _ensure_layer_batched_keys(keys)
    kmeans_dtype = _resolve_kmeans_dtype(kmeans_dtype)
    _validate_cluster_axis(cluster_axis)

    if kmeans_mode not in {"concat_heads", "avg_heads", "per_head"}:
        raise ValueError(
            "kmeans_mode must be one of 'concat_heads', 'avg_heads', or "
            "'per_head'."
        )

    decompose_fn, decomp_kwargs = _get_decomposition(
        decomposition_method=decomposition_method,
        rank_selection=rank_selection,
        comp_ratio=comp_ratio,
        energy_threshold=energy_threshold,
        decomp_n_iter=decomp_n_iter,
        decomp_lr=decomp_lr,
    )

    results = []
    for layer_idx in range(keys.size(0)):
        prefix_keys, compressed_len = _select_prefix(
            keys[layer_idx],
            prefix_end=prefix_end,
            local_window=local_window,
        )
        batch_size, num_heads, seq_len, head_dim = prefix_keys.shape

        if kmeans_mode == "per_head":
            base_tensor = prefix_keys.reshape(
                batch_size * num_heads,
                seq_len,
                head_dim,
            )
            (
                _,
                grouped_items,
                assignments,
                segment_ranges,
                cluster_count,
            ) = _cluster_items_and_ranges(
                base_tensor,
                n_clusters,
                kmeans_n_iter,
                kmeans_init,
                kmeans_dtype,
                kmeans_cluster_size=kmeans_cluster_size,
                cluster_axis=cluster_axis,
            )
            if cluster_axis == "rows":
                grouped_target = grouped_items
            else:
                grouped_target, _, _ = _group_tensor_last_dim_by_cluster(
                    base_tensor,
                    assignments,
                )
            scatter_metrics = _scatter_from_grouped(
                grouped_items,
                segment_ranges,
            )
            reconstructed = _reconstruct_lr_segments(
                grouped_target,
                segment_ranges,
                decompose_fn,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            recon_metrics = _low_rank_recon_metrics(
                grouped_target,
                reconstructed,
            )
            metrics = _merge_metric_lists(scatter_metrics, recon_metrics)
            for flat_idx, metric in enumerate(metrics):
                batch_idx = flat_idx // num_heads
                head_idx = flat_idx % num_heads
                results.append(
                    {
                        "cache_type": "kmeans_lr",
                        "cluster_axis": cluster_axis,
                        "kmeans_mode": kmeans_mode,
                        "metric_scope": "head",
                        "layer_idx": layer_idx,
                        "batch_idx": batch_idx,
                        "head_idx": head_idx,
                        "seq_len": seq_len,
                        "compressed_len": compressed_len,
                        "cluster_count": cluster_count,
                        **metric,
                    }
                )
            continue

        if kmeans_mode == "avg_heads":
            token_features = prefix_keys.mean(dim=1)
        else:
            token_features = prefix_keys.transpose(1, 2).reshape(
                batch_size,
                seq_len,
                -1,
            )

        (
            _,
            grouped_items,
            assignments,
            segment_ranges,
            cluster_count,
        ) = _cluster_items_and_ranges(
            token_features,
            n_clusters,
            kmeans_n_iter,
            kmeans_init,
            kmeans_dtype,
            kmeans_cluster_size=kmeans_cluster_size,
            cluster_axis=cluster_axis,
        )

        if cluster_axis == "rows":
            grouped_target, _, _ = group_keys_by_cluster(prefix_keys, assignments)
            scatter_metrics = _scatter_from_grouped(
                grouped_items,
                segment_ranges,
            )
            reconstructed = _reconstruct_lr_segments(
                grouped_target,
                segment_ranges,
                decompose_fn,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            layer_recon_metrics = _low_rank_recon_metrics(
                grouped_target,
                reconstructed,
            )
            layer_metrics = _merge_metric_lists(scatter_metrics, layer_recon_metrics)
            for batch_idx, metric in enumerate(layer_metrics):
                results.append(
                    {
                        "cache_type": "kmeans_lr",
                        "cluster_axis": cluster_axis,
                        "kmeans_mode": kmeans_mode,
                        "metric_scope": "layer",
                        "layer_idx": layer_idx,
                        "batch_idx": batch_idx,
                        "head_idx": None,
                        "seq_len": seq_len,
                        "compressed_len": compressed_len,
                        "cluster_count": cluster_count,
                        **metric,
                    }
                )

            if include_head_breakdown:
                for head_idx in range(num_heads):
                    head_scatter_metrics = _scatter_from_grouped(
                        grouped_target[:, head_idx],
                        segment_ranges,
                    )
                    head_recon_metrics = _low_rank_recon_metrics(
                        grouped_target[:, head_idx],
                        reconstructed[:, head_idx],
                    )
                    head_metrics = _merge_metric_lists(
                        head_scatter_metrics,
                        head_recon_metrics,
                    )
                    for batch_idx, metric in enumerate(head_metrics):
                        results.append(
                            {
                                "cache_type": "kmeans_lr",
                                "cluster_axis": cluster_axis,
                                "kmeans_mode": kmeans_mode,
                                "metric_scope": "head",
                                "layer_idx": layer_idx,
                                "batch_idx": batch_idx,
                                "head_idx": head_idx,
                                "seq_len": seq_len,
                                "compressed_len": compressed_len,
                                "cluster_count": cluster_count,
                                **metric,
                            }
                        )
            continue

        grouped_target, _, inverse_permutation = _group_tensor_last_dim_by_cluster(
            token_features,
            assignments,
        )
        scatter_metrics = _scatter_from_grouped(
            grouped_items,
            segment_ranges,
        )
        reconstructed = _reconstruct_lr_segments(
            grouped_target,
            segment_ranges,
            decompose_fn,
            decomp_kwargs,
            cluster_axis=cluster_axis,
        )
        restored_reconstructed = _restore_tensor_last_dim(
            reconstructed,
            inverse_permutation,
        )
        layer_recon_metrics = _low_rank_recon_metrics(
            grouped_target,
            reconstructed,
        )
        layer_metrics = _merge_metric_lists(scatter_metrics, layer_recon_metrics)
        for batch_idx, metric in enumerate(layer_metrics):
            results.append(
                {
                    "cache_type": "kmeans_lr",
                    "cluster_axis": cluster_axis,
                    "kmeans_mode": kmeans_mode,
                    "metric_scope": "layer",
                    "layer_idx": layer_idx,
                    "batch_idx": batch_idx,
                    "head_idx": None,
                    "seq_len": seq_len,
                    "compressed_len": compressed_len,
                    "cluster_count": cluster_count,
                    **metric,
                }
            )

        if include_head_breakdown:
            for head_idx in range(num_heads):
                if kmeans_mode == "avg_heads":
                    head_target = prefix_keys[:, head_idx]
                    head_assignments = assignments
                    grouped_head_target, _, _ = _group_tensor_last_dim_by_cluster(
                        head_target,
                        head_assignments,
                    )
                    grouped_head_items, head_segment_ranges = _group_features_and_ranges(
                        head_target.transpose(1, 2).contiguous(),
                        head_assignments,
                        cluster_count,
                    )
                    head_reconstructed = _reconstruct_lr_segments(
                        grouped_head_target,
                        head_segment_ranges,
                        decompose_fn,
                        decomp_kwargs,
                        cluster_axis=cluster_axis,
                    )
                    head_recon_metrics = _low_rank_recon_metrics(
                        grouped_head_target,
                        head_reconstructed,
                    )
                    head_scatter_metrics = _scatter_from_grouped(
                        grouped_head_items,
                        head_segment_ranges,
                    )
                else:
                    start_idx = head_idx * head_dim
                    end_idx = start_idx + head_dim
                    head_items, head_segment_ranges = _group_features_and_ranges(
                        token_features[..., start_idx:end_idx]
                        .transpose(1, 2)
                        .contiguous(),
                        assignments[..., start_idx:end_idx],
                        cluster_count,
                    )
                    head_scatter_metrics = _scatter_from_grouped(
                        head_items,
                        head_segment_ranges,
                    )
                    head_recon_metrics = _low_rank_recon_metrics(
                        token_features[..., start_idx:end_idx],
                        restored_reconstructed[..., start_idx:end_idx],
                    )

                head_metrics = _merge_metric_lists(
                    head_scatter_metrics,
                    head_recon_metrics,
                )
                for batch_idx, metric in enumerate(head_metrics):
                    results.append(
                        {
                            "cache_type": "kmeans_lr",
                            "cluster_axis": cluster_axis,
                            "kmeans_mode": kmeans_mode,
                            "metric_scope": "head",
                            "layer_idx": layer_idx,
                            "batch_idx": batch_idx,
                            "head_idx": head_idx,
                            "seq_len": seq_len,
                            "compressed_len": compressed_len,
                            "cluster_count": cluster_count,
                            **metric,
                        }
                    )

    return results


def _get_group_bounds(layer_idx, layer_group_size, num_layers=None):
    group_start = (layer_idx // layer_group_size) * layer_group_size
    group_last = group_start + layer_group_size - 1
    if num_layers is not None:
        group_last = min(group_last, num_layers - 1)
    return group_start, group_last


def analyze_kmeans_xkv(
    keys,
    layer_group_size=2,
    num_layers=None,
    n_clusters=8,
    kmeans_cluster_size=None,
    kmeans_n_iter=8,
    kmeans_init="infllm",
    kmeans_dtype=torch.float32,
    prefix_end=None,
    local_window=0,
    decomposition_method="svd",
    rank_selection="comp_ratio",
    comp_ratio=2.0,
    energy_threshold=0.95,
    decomp_n_iter=3,
    decomp_lr=1e-2,
    cluster_axis="rows",
):
    keys = _ensure_layer_batched_keys(keys)
    kmeans_dtype = _resolve_kmeans_dtype(kmeans_dtype)
    _validate_cluster_axis(cluster_axis)

    if layer_group_size <= 0:
        raise ValueError("layer_group_size must be positive.")
    if num_layers is None:
        num_layers = keys.size(0)
    if num_layers <= 0 or num_layers > keys.size(0):
        raise ValueError("num_layers must be in [1, keys.size(0)].")

    if decomposition_method != "svd":
        raise NotImplementedError(
            "KMeansXKVKeysCache-style grouped reconstruction currently "
            "supports decomposition_method='svd' only."
        )

    _, decomp_kwargs = _get_decomposition(
        decomposition_method=decomposition_method,
        rank_selection=rank_selection,
        comp_ratio=comp_ratio,
        energy_threshold=energy_threshold,
        decomp_n_iter=decomp_n_iter,
        decomp_lr=decomp_lr,
    )

    results = []
    for layer_idx in range(num_layers):
        group_start, group_last = _get_group_bounds(
            layer_idx,
            layer_group_size,
            num_layers,
        )
        if layer_idx != group_last:
            continue

        group_tensors = [keys[i] for i in range(group_start, group_last + 1)]
        seq_len = group_tensors[-1].size(-2)
        if any(tensor.size(-2) != seq_len for tensor in group_tensors[:-1]):
            raise ValueError(
                "All layers in an xKV group must share the same cached length."
            )

        prefix_tensors = []
        split_sizes = []
        compressed_len = None
        for tensor in group_tensors:
            prefix_tensor, compressed_len = _select_prefix(
                tensor,
                prefix_end=prefix_end,
                local_window=local_window,
            )
            prefix_flat = prefix_tensor.transpose(1, 2).reshape(
                prefix_tensor.size(0),
                prefix_tensor.size(-2),
                -1,
            )
            prefix_tensors.append(prefix_flat)
            split_sizes.append(prefix_flat.size(-1))

        group_prefix = torch.cat(prefix_tensors, dim=-1)
        (
            _,
            grouped_items,
            assignments,
            segment_ranges,
            cluster_count,
        ) = _cluster_items_and_ranges(
            group_prefix,
            n_clusters,
            kmeans_n_iter,
            kmeans_init,
            kmeans_dtype,
            kmeans_cluster_size=kmeans_cluster_size,
            cluster_axis=cluster_axis,
        )

        if cluster_axis == "rows":
            grouped_target = grouped_items
            reconstructed_group = _reconstruct_xkv_segments(
                grouped_target,
                segment_ranges,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            grouped_layer_targets = torch.split(
                grouped_target,
                split_sizes,
                dim=-1,
            )
            reconstructed_layer_targets = torch.split(
                reconstructed_group,
                split_sizes,
                dim=-1,
            )
        else:
            grouped_target, _, inverse_permutation = _group_tensor_last_dim_by_cluster(
                group_prefix,
                assignments,
            )
            reconstructed_group = _reconstruct_xkv_segments(
                grouped_target,
                segment_ranges,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            restored_reconstructed = _restore_tensor_last_dim(
                reconstructed_group,
                inverse_permutation,
            )

        group_scatter_metrics = _scatter_from_grouped(
            grouped_items,
            segment_ranges,
        )
        group_recon_metrics = _low_rank_recon_metrics(
            grouped_target,
            reconstructed_group,
        )
        group_metrics = _merge_metric_lists(
            group_scatter_metrics,
            group_recon_metrics,
        )
        for batch_idx, metric in enumerate(group_metrics):
            results.append(
                {
                    "cache_type": "kmeans_xkv",
                    "cluster_axis": cluster_axis,
                    "metric_scope": "group",
                    "layer_idx": group_last,
                    "group_start_layer": group_start,
                    "group_last_layer": group_last,
                    "batch_idx": batch_idx,
                    "head_idx": None,
                    "seq_len": group_prefix.size(1),
                    "compressed_len": compressed_len,
                    "cluster_count": cluster_count,
                    **metric,
                }
            )

        col_offset = 0
        for offset, split_size in enumerate(split_sizes):
            actual_layer_idx = group_start + offset
            if cluster_axis == "rows":
                layer_target = grouped_layer_targets[offset]
                layer_recon = reconstructed_layer_targets[offset]
                layer_scatter_metrics = _scatter_from_grouped(
                    layer_target,
                    segment_ranges,
                )
                layer_recon_metrics = _low_rank_recon_metrics(
                    layer_target,
                    layer_recon,
                )
            else:
                layer_original = group_prefix[..., col_offset : col_offset + split_size]
                layer_assignments = assignments[
                    ..., col_offset : col_offset + split_size
                ]
                layer_items, layer_segment_ranges = _group_features_and_ranges(
                    layer_original.transpose(1, 2).contiguous(),
                    layer_assignments,
                    cluster_count,
                )
                layer_scatter_metrics = _scatter_from_grouped(
                    layer_items,
                    layer_segment_ranges,
                )
                layer_recon_metrics = _low_rank_recon_metrics(
                    layer_original,
                    restored_reconstructed[
                        ..., col_offset : col_offset + split_size
                    ],
                )
                col_offset += split_size

            layer_metrics = _merge_metric_lists(
                layer_scatter_metrics,
                layer_recon_metrics,
            )
            for batch_idx, metric in enumerate(layer_metrics):
                results.append(
                    {
                        "cache_type": "kmeans_xkv",
                        "cluster_axis": cluster_axis,
                        "metric_scope": "layer",
                        "layer_idx": actual_layer_idx,
                        "group_start_layer": group_start,
                        "group_last_layer": group_last,
                        "batch_idx": batch_idx,
                        "head_idx": None,
                        "seq_len": group_prefix.size(1),
                        "compressed_len": compressed_len,
                        "cluster_count": cluster_count,
                        **metric,
                    }
                )

    return results


# Cluster rows

In [10]:
kmeans_cfg = {
    "n_clusters": 8,
    "kmeans_cluster_size": 512,
    "kmeans_n_iter": 8,
    "kmeans_init": "infllm",
    "kmeans_dtype": torch.float32,
}

decomposition_cfg = {
    "decomposition_method": "svd",
    "rank_selection": "comp_ratio",
    "comp_ratio": 2.0,
    "energy_threshold": 0.95,
    "decomp_n_iter": 3,
    "decomp_lr": 1e-2,
}

lrk_mode = "per_head"  # "avg_heads" or "per_head"
cluster_axis = "rows"  # "rows" or "cols"
include_lrk_head_breakdown = False
prefix_end = prompt_len
local_window = 0

xkv_layer_group_size = 4
xkv_num_layers = keys.size(0)


### Keys

In [ ]:
lrk_results = analyze_kmeans_lrk(
    keys,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg,
    **decomposition_cfg,
)

xkv_results = analyze_kmeans_xkv(
    keys,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg,
    **decomposition_cfg,
)

kmeans_cfg_n1 = {**kmeans_cfg, "n_clusters": 1, "kmeans_cluster_size": None}

lrk_results_n1 = analyze_kmeans_lrk(
    keys,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

xkv_results_n1 = analyze_kmeans_xkv(
    keys,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("First LRK result:", lrk_results[0] if lrk_results else None)
    print("First xKV result:", xkv_results[0] if xkv_results else None)
    print("First LRK n=1 result:", lrk_results_n1[0] if lrk_results_n1 else None)
    print("First xKV n=1 result:", xkv_results_n1[0] if xkv_results_n1 else None)
else:
    cols_to_hide = ["batch_idx", "seq_len", "compressed_len"]
    lrk_df = pd.DataFrame(lrk_results).drop(columns=cols_to_hide, errors="ignore")
    xkv_df = pd.DataFrame(xkv_results).drop(columns=cols_to_hide, errors="ignore")
    lrk_df_n1 = pd.DataFrame(lrk_results_n1).drop(columns=cols_to_hide, errors="ignore")
    xkv_df_n1 = pd.DataFrame(xkv_results_n1).drop(columns=cols_to_hide, errors="ignore")

    def build_summary(label, df):
        return {
            "case": label,
            "rows": len(df),
            "mean_eta": df["eta"].mean(),
            "median_eta": df["eta"].median(),
            "mean_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].mean(),
            "median_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].median(),
            "min_eta": df["eta"].min(),
            "max_eta": df["eta"].max(),
        }

    summary_df = pd.DataFrame(
        [
            build_summary(f"lrk_{cluster_axis}", lrk_df),
            build_summary(f"xkv_{cluster_axis}", xkv_df),
            build_summary(f"lrk_{cluster_axis}_n_clusters_1", lrk_df_n1),
            build_summary(f"xkv_{cluster_axis}_n_clusters_1", xkv_df_n1),
        ]
    )
    display(summary_df)
    # display(lrk_df)
    # display(xkv_df)
    # display(lrk_df_n1)
    # display(xkv_df_n1)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,lrk_rows,256,0.235341,0.225570,0.137055,0.143355,0.035008,0.485646
1,xkv_rows,40,0.242459,0.246324,0.092277,0.094444,0.115809,0.381579
2,lrk_rows_n_clusters_1,256,0.315100,0.304945,0.146556,0.153671,0.074367,0.558442
3,xkv_rows_n_clusters_1,40,0.312275,0.306936,0.070334,0.072464,0.173713,0.473684


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,rows,per_head,head,0,0,7,139264.0,0.092391,40.250,0.032884
1,kmeans_lr,rows,per_head,head,0,1,7,134144.0,0.113715,34.000,0.031250
2,kmeans_lr,rows,per_head,head,0,2,7,180224.0,0.167939,42.500,0.041182
3,kmeans_lr,rows,per_head,head,0,3,7,90112.0,0.132530,25.625,0.031098
4,kmeans_lr,rows,per_head,head,0,4,7,69632.0,0.103659,16.125,0.019665
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,rows,per_head,head,31,3,7,356352.0,0.124286,187.000,0.110259
252,kmeans_lr,rows,per_head,head,31,4,7,344064.0,0.211055,187.000,0.146094
253,kmeans_lr,rows,per_head,head,31,5,7,448512.0,0.252304,187.000,0.139970
254,kmeans_lr,rows,per_head,head,31,6,7,305152.0,0.120942,167.000,0.104899


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,rows,group,3,0,3,None,7,10616832.0,0.176087,588.0,0.075617
1,kmeans_xkv,rows,layer,0,0,3,None,7,1286144.0,0.152132,189.0,0.064904
2,kmeans_xkv,rows,layer,1,0,3,None,7,2064384.0,0.115809,191.0,0.045218
3,kmeans_xkv,rows,layer,2,0,3,None,7,4390912.0,0.250000,352.0,0.083969
4,kmeans_xkv,rows,layer,3,0,3,None,7,2834432.0,0.173000,388.0,0.095850
5,kmeans_xkv,rows,group,7,4,7,None,7,15663104.0,0.203231,824.0,0.093978
6,kmeans_xkv,rows,layer,4,4,7,None,7,2932736.0,0.168233,386.0,0.092788
7,kmeans_xkv,rows,layer,5,4,7,None,7,4194304.0,0.228571,414.0,0.096549
8,kmeans_xkv,rows,layer,6,4,7,None,7,3964928.0,0.201667,414.0,0.093076
9,kmeans_xkv,rows,layer,7,4,7,None,7,4521984.0,0.209091,432.0,0.093103


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,rows,per_head,head,0,0,1,235520.0,0.156250,45.500,0.037173
1,kmeans_lr,rows,per_head,head,0,1,1,260096.0,0.220486,42.250,0.038833
2,kmeans_lr,rows,per_head,head,0,2,1,423936.0,0.395038,54.250,0.052568
3,kmeans_lr,rows,per_head,head,0,3,1,189440.0,0.278614,31.750,0.038532
4,kmeans_lr,rows,per_head,head,0,4,1,172032.0,0.256098,19.125,0.023323
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,rows,per_head,head,31,3,1,540672.0,0.188571,209.000,0.123231
252,kmeans_lr,rows,per_head,head,31,4,1,458752.0,0.281407,196.000,0.153125
253,kmeans_lr,rows,per_head,head,31,5,1,626688.0,0.352535,195.000,0.145958
254,kmeans_lr,rows,per_head,head,31,6,1,462848.0,0.183442,187.000,0.117462


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,rows,group,3,0,3,None,1,14483456.0,0.240217,394.0,0.050669
1,kmeans_xkv,rows,layer,0,0,3,None,1,2244608.0,0.265504,138.0,0.047390
2,kmeans_xkv,rows,layer,1,0,3,None,1,3096576.0,0.173713,157.0,0.037169
3,kmeans_xkv,rows,layer,2,0,3,None,1,5275648.0,0.300373,223.0,0.053197
4,kmeans_xkv,rows,layer,3,0,3,None,1,3850240.0,0.234064,250.0,0.061759
5,kmeans_xkv,rows,group,7,4,7,None,1,20447232.0,0.265306,636.0,0.072536
6,kmeans_xkv,rows,layer,4,4,7,None,1,4096000.0,0.236742,318.0,0.076442
7,kmeans_xkv,rows,layer,5,4,7,None,1,5308416.0,0.289286,310.0,0.072295
8,kmeans_xkv,rows,layer,6,4,7,None,1,5242880.0,0.266667,322.0,0.072392
9,kmeans_xkv,rows,layer,7,4,7,None,1,5865472.0,0.269578,324.0,0.069349


### Values

In [12]:
lrk_results = analyze_kmeans_lrk(
    values,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg,
    **decomposition_cfg,
)

xkv_results = analyze_kmeans_xkv(
    values,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg,
    **decomposition_cfg,
)

kmeans_cfg_n1 = {**kmeans_cfg, "n_clusters": 1, "kmeans_cluster_size": None}

lrk_results_n1 = analyze_kmeans_lrk(
    values,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

xkv_results_n1 = analyze_kmeans_xkv(
    values,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("First LRK result:", lrk_results[0] if lrk_results else None)
    print("First xKV result:", xkv_results[0] if xkv_results else None)
    print("First LRK n=1 result:", lrk_results_n1[0] if lrk_results_n1 else None)
    print("First xKV n=1 result:", xkv_results_n1[0] if xkv_results_n1 else None)
else:
    cols_to_hide = ["batch_idx", "seq_len", "compressed_len"]
    lrk_df = pd.DataFrame(lrk_results).drop(columns=cols_to_hide, errors="ignore")
    xkv_df = pd.DataFrame(xkv_results).drop(columns=cols_to_hide, errors="ignore")
    lrk_df_n1 = pd.DataFrame(lrk_results_n1).drop(columns=cols_to_hide, errors="ignore")
    xkv_df_n1 = pd.DataFrame(xkv_results_n1).drop(columns=cols_to_hide, errors="ignore")

    def build_summary(label, df):
        return {
            "case": label,
            "rows": len(df),
            "mean_eta": df["eta"].mean(),
            "median_eta": df["eta"].median(),
            "mean_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].mean(),
            "median_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].median(),
            "min_eta": df["eta"].min(),
            "max_eta": df["eta"].max(),
        }

    summary_df = pd.DataFrame(
        [
            build_summary(f"lrk_{cluster_axis}", lrk_df),
            build_summary(f"xkv_{cluster_axis}", xkv_df),
            build_summary(f"lrk_{cluster_axis}_n_clusters_1", lrk_df_n1),
            build_summary(f"xkv_{cluster_axis}_n_clusters_1", xkv_df_n1),
        ]
    )
    display(summary_df)
    # display(lrk_df)
    # display(xkv_df)
    # display(lrk_df_n1)
    # display(xkv_df_n1)

,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,lrk_rows,256,0.796892,0.820140,0.408540,0.410670,0.140944,0.959391
1,xkv_rows,40,0.841087,0.852803,0.253372,0.254087,0.471939,0.909639
2,lrk_rows_n_clusters_1,256,0.890863,0.923303,0.452579,0.458740,0.213010,0.989011
3,xkv_rows_n_clusters_1,40,0.884039,0.903943,0.217436,0.213246,0.528061,0.939560


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,rows,per_head,head,0,0,7,576.0,0.774194,5.50000,0.201835
1,kmeans_lr,rows,per_head,head,0,1,7,728.0,0.943005,13.25000,0.477477
2,kmeans_lr,rows,per_head,head,0,2,7,512.0,0.901408,6.78125,0.284031
3,kmeans_lr,rows,per_head,head,0,3,7,448.0,0.881890,9.31250,0.413889
4,kmeans_lr,rows,per_head,head,0,4,7,400.0,0.873362,6.93750,0.324561
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,rows,per_head,head,31,3,7,113152.0,0.140944,139.00000,0.155134
252,kmeans_lr,rows,per_head,head,31,4,7,68096.0,0.791667,114.50000,0.389456
253,kmeans_lr,rows,per_head,head,31,5,7,151552.0,0.902439,201.00000,0.490244
254,kmeans_lr,rows,per_head,head,31,6,7,101888.0,0.667785,135.00000,0.346154


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,rows,group,3,0,3,None,7,259072.0,0.878472,108.00,0.198529
1,kmeans_xkv,rows,layer,0,0,3,None,7,4480.0,0.843373,24.25,0.332192
2,kmeans_xkv,rows,layer,1,0,3,None,7,20864.0,0.895604,37.00,0.241830
3,kmeans_xkv,rows,layer,2,0,3,None,7,88576.0,0.843902,64.50,0.199074
4,kmeans_xkv,rows,layer,3,0,3,None,7,146432.0,0.905063,74.00,0.184080
5,kmeans_xkv,rows,group,7,4,7,None,7,593920.0,0.838150,233.00,0.277381
6,kmeans_xkv,rows,layer,4,4,7,None,7,126464.0,0.845890,118.00,0.305699
7,kmeans_xkv,rows,layer,5,4,7,None,7,126976.0,0.815789,109.00,0.276650
8,kmeans_xkv,rows,layer,6,4,7,None,7,155648.0,0.863636,118.50,0.279481
9,kmeans_xkv,rows,layer,7,4,7,None,7,185344.0,0.826484,120.00,0.253165


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,rows,per_head,head,0,0,1,712.0,0.956989,9.4375,0.346330
1,kmeans_lr,rows,per_head,head,0,1,1,752.0,0.969072,14.3750,0.518018
2,kmeans_lr,rows,per_head,head,0,2,1,560.0,0.985915,8.9375,0.374346
3,kmeans_lr,rows,per_head,head,0,3,1,488.0,0.960630,10.5625,0.469444
4,kmeans_lr,rows,per_head,head,0,4,1,448.0,0.982456,8.1875,0.383041
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,rows,per_head,head,31,3,1,171008.0,0.213010,153.0000,0.170759
252,kmeans_lr,rows,per_head,head,31,4,1,77824.0,0.904762,133.0000,0.452381
253,kmeans_lr,rows,per_head,head,31,5,1,156672.0,0.932927,210.0000,0.512195
254,kmeans_lr,rows,per_head,head,31,6,1,123904.0,0.812081,154.0000,0.394872


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,rows,group,3,0,3,None,1,270336.0,0.916667,95.500,0.175551
1,kmeans_xkv,rows,layer,0,0,3,None,1,4672.0,0.879518,26.875,0.368151
2,kmeans_xkv,rows,layer,1,0,3,None,1,21888.0,0.939560,39.500,0.258170
3,kmeans_xkv,rows,layer,2,0,3,None,1,92672.0,0.882927,58.500,0.180556
4,kmeans_xkv,rows,layer,3,0,3,None,1,150528.0,0.930380,58.750,0.146144
5,kmeans_xkv,rows,group,7,4,7,None,1,651264.0,0.919075,193.000,0.229762
6,kmeans_xkv,rows,layer,4,4,7,None,1,137216.0,0.917808,101.000,0.261658
7,kmeans_xkv,rows,layer,5,4,7,None,1,139264.0,0.894737,94.500,0.239848
8,kmeans_xkv,rows,layer,6,4,7,None,1,167936.0,0.931818,96.500,0.227594
9,kmeans_xkv,rows,layer,7,4,7,None,1,204800.0,0.913242,95.500,0.201477


## Cluster cols

In [13]:
kmeans_cfg = {
    "n_clusters": 8,
    "kmeans_cluster_size": 512,
    "kmeans_n_iter": 8,
    "kmeans_init": "infllm",
    "kmeans_dtype": torch.float32,
}

decomposition_cfg = {
    "decomposition_method": "svd",
    "rank_selection": "comp_ratio",
    "comp_ratio": 2.0,
    "energy_threshold": 0.95,
    "decomp_n_iter": 3,
    "decomp_lr": 1e-2,
}

lrk_mode = "per_head"  # "avg_heads" or "per_head"
cluster_axis = "cols"  # "rows" or "cols"
include_lrk_head_breakdown = False
prefix_end = prompt_len
local_window = 0

xkv_layer_group_size = 4
xkv_num_layers = keys.size(0)


### Keys

In [14]:
lrk_results = analyze_kmeans_lrk(
    keys,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg,
    **decomposition_cfg,
)

xkv_results = analyze_kmeans_xkv(
    keys,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg,
    **decomposition_cfg,
)

kmeans_cfg_n1 = {**kmeans_cfg, "n_clusters": 1, "kmeans_cluster_size": None}

lrk_results_n1 = analyze_kmeans_lrk(
    keys,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

xkv_results_n1 = analyze_kmeans_xkv(
    keys,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("First LRK result:", lrk_results[0] if lrk_results else None)
    print("First xKV result:", xkv_results[0] if xkv_results else None)
    print("First LRK n=1 result:", lrk_results_n1[0] if lrk_results_n1 else None)
    print("First xKV n=1 result:", xkv_results_n1[0] if xkv_results_n1 else None)
else:
    cols_to_hide = ["batch_idx", "seq_len", "compressed_len"]
    lrk_df = pd.DataFrame(lrk_results).drop(columns=cols_to_hide, errors="ignore")
    xkv_df = pd.DataFrame(xkv_results).drop(columns=cols_to_hide, errors="ignore")
    lrk_df_n1 = pd.DataFrame(lrk_results_n1).drop(columns=cols_to_hide, errors="ignore")
    xkv_df_n1 = pd.DataFrame(xkv_results_n1).drop(columns=cols_to_hide, errors="ignore")

    def build_summary(label, df):
        return {
            "case": label,
            "rows": len(df),
            "mean_eta": df["eta"].mean(),
            "median_eta": df["eta"].median(),
            "mean_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].mean(),
            "median_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].median(),
            "min_eta": df["eta"].min(),
            "max_eta": df["eta"].max(),
        }

    summary_df = pd.DataFrame(
        [
            build_summary(f"lrk_{cluster_axis}", lrk_df),
            build_summary(f"xkv_{cluster_axis}", xkv_df),
            build_summary(f"lrk_{cluster_axis}_n_clusters_1", lrk_df_n1),
            build_summary(f"xkv_{cluster_axis}_n_clusters_1", xkv_df_n1),
        ]
    )
    display(summary_df)
    # display(lrk_df)
    # display(xkv_df)
    # display(lrk_df_n1)
    # display(xkv_df_n1)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,lrk_cols,256,0.992535,0.993506,0.146559,0.153671,0.932039,1.000000
1,xkv_cols,40,0.369702,0.367856,0.102440,0.106618,0.209854,0.493421
2,lrk_cols_n_clusters_1,256,0.992535,0.993506,0.146559,0.153671,0.932039,1.000000
3,xkv_cols_n_clusters_1,40,0.998950,1.000000,0.070360,0.072464,0.992481,1.000000


### Values

In [15]:
lrk_results = analyze_kmeans_lrk(
    values,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg,
    **decomposition_cfg,
)

xkv_results = analyze_kmeans_xkv(
    values,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg,
    **decomposition_cfg,
)

kmeans_cfg_n1 = {**kmeans_cfg, "n_clusters": 1, "kmeans_cluster_size": None}

lrk_results_n1 = analyze_kmeans_lrk(
    values,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

xkv_results_n1 = analyze_kmeans_xkv(
    values,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("First LRK result:", lrk_results[0] if lrk_results else None)
    print("First xKV result:", xkv_results[0] if xkv_results else None)
    print("First LRK n=1 result:", lrk_results_n1[0] if lrk_results_n1 else None)
    print("First xKV n=1 result:", xkv_results_n1[0] if xkv_results_n1 else None)
else:
    cols_to_hide = ["batch_idx", "seq_len", "compressed_len"]
    lrk_df = pd.DataFrame(lrk_results).drop(columns=cols_to_hide, errors="ignore")
    xkv_df = pd.DataFrame(xkv_results).drop(columns=cols_to_hide, errors="ignore")
    lrk_df_n1 = pd.DataFrame(lrk_results_n1).drop(columns=cols_to_hide, errors="ignore")
    xkv_df_n1 = pd.DataFrame(xkv_results_n1).drop(columns=cols_to_hide, errors="ignore")

    def build_summary(label, df):
        return {
            "case": label,
            "rows": len(df),
            "mean_eta": df["eta"].mean(),
            "median_eta": df["eta"].median(),
            "mean_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].mean(),
            "median_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].median(),
            "min_eta": df["eta"].min(),
            "max_eta": df["eta"].max(),
        }

    summary_df = pd.DataFrame(
        [
            build_summary(f"lrk_{cluster_axis}", lrk_df),
            build_summary(f"xkv_{cluster_axis}", xkv_df),
            build_summary(f"lrk_{cluster_axis}_n_clusters_1", lrk_df_n1),
            build_summary(f"xkv_{cluster_axis}_n_clusters_1", xkv_df_n1),
        ]
    )
    display(summary_df)
    # display(lrk_df)
    # display(xkv_df)
    # display(lrk_df_n1)
    # display(xkv_df_n1)

,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,lrk_cols,256,0.991860,0.992308,0.452579,0.458740,0.962963,1.000000
1,xkv_cols,40,0.887461,0.901931,0.301894,0.303222,0.589286,0.934066
2,lrk_cols_n_clusters_1,256,0.991860,0.992308,0.452579,0.458740,0.962963,1.000000
3,xkv_cols_n_clusters_1,40,0.999150,1.000000,0.217347,0.213246,0.993827,1.000000


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,cols,per_head,head,0,0,1,736.0,0.989247,9.4375,0.346330
1,kmeans_lr,cols,per_head,head,0,1,1,768.0,0.989691,14.3750,0.518018
2,kmeans_lr,cols,per_head,head,0,2,1,564.0,0.992958,8.9375,0.374346
3,kmeans_lr,cols,per_head,head,0,3,1,504.0,0.992126,10.5625,0.469444
4,kmeans_lr,cols,per_head,head,0,4,1,454.0,0.991266,8.1875,0.383041
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,cols,per_head,head,31,3,1,802816.0,1.000000,153.0000,0.170759
252,kmeans_lr,cols,per_head,head,31,4,1,84992.0,0.988095,133.0000,0.452381
253,kmeans_lr,cols,per_head,head,31,5,1,166912.0,0.993902,210.0000,0.512195
254,kmeans_lr,cols,per_head,head,31,6,1,151552.0,0.993289,154.0000,0.394872


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,cols,group,3,0,3,None,8,268288.0,0.909722,139.000,0.255515
1,kmeans_xkv,cols,layer,0,0,3,None,8,4768.0,0.897590,28.625,0.392123
2,kmeans_xkv,cols,layer,1,0,3,None,8,21760.0,0.934066,47.750,0.312092
3,kmeans_xkv,cols,layer,2,0,3,None,8,93184.0,0.887805,87.000,0.268519
4,kmeans_xkv,cols,layer,3,0,3,None,8,146432.0,0.905063,92.500,0.230100
5,kmeans_xkv,cols,group,7,4,7,None,8,638976.0,0.901734,256.000,0.304762
6,kmeans_xkv,cols,layer,4,4,7,None,8,134144.0,0.897260,130.000,0.336788
7,kmeans_xkv,cols,layer,5,4,7,None,8,137216.0,0.881579,129.000,0.327411
8,kmeans_xkv,cols,layer,6,4,7,None,8,163840.0,0.909091,125.500,0.295991
9,kmeans_xkv,cols,layer,7,4,7,None,8,198656.0,0.885845,128.000,0.270042


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,cols,per_head,head,0,0,1,736.0,0.989247,9.4375,0.346330
1,kmeans_lr,cols,per_head,head,0,1,1,768.0,0.989691,14.3750,0.518018
2,kmeans_lr,cols,per_head,head,0,2,1,564.0,0.992958,8.9375,0.374346
3,kmeans_lr,cols,per_head,head,0,3,1,504.0,0.992126,10.5625,0.469444
4,kmeans_lr,cols,per_head,head,0,4,1,454.0,0.991266,8.1875,0.383041
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,cols,per_head,head,31,3,1,802816.0,1.000000,153.0000,0.170759
252,kmeans_lr,cols,per_head,head,31,4,1,84992.0,0.988095,133.0000,0.452381
253,kmeans_lr,cols,per_head,head,31,5,1,166912.0,0.993902,210.0000,0.512195
254,kmeans_lr,cols,per_head,head,31,6,1,151552.0,0.993289,154.0000,0.394872


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,cols,group,3,0,3,None,1,294912.0,1.000000,95.500,0.175551
1,kmeans_xkv,cols,layer,0,0,3,None,1,5312.0,1.000000,26.875,0.368151
2,kmeans_xkv,cols,layer,1,0,3,None,1,23168.0,1.000000,39.500,0.258170
3,kmeans_xkv,cols,layer,2,0,3,None,1,104960.0,1.000000,58.500,0.180556
4,kmeans_xkv,cols,layer,3,0,3,None,1,161792.0,1.000000,58.750,0.146144
5,kmeans_xkv,cols,group,7,4,7,None,1,708608.0,1.000000,193.000,0.229762
6,kmeans_xkv,cols,layer,4,4,7,None,1,149504.0,1.000000,101.000,0.261658
7,kmeans_xkv,cols,layer,5,4,7,None,1,155648.0,1.000000,94.500,0.239848
8,kmeans_xkv,cols,layer,6,4,7,None,1,180224.0,1.000000,97.000,0.228774
9,kmeans_xkv,cols,layer,7,4,7,None,1,224256.0,1.000000,95.500,0.201477
